# Domain-Adapting Qwen2.5-1.5B-Instruct Using QLoRA

## Objective

The objective of this project is to adapt a small open-weight language model to the coding domain using QLoRA (Quantized Low-Rank Adaptation).

Instead of training all 1.55 billion parameters, LoRA trains only a small set of adapter parameters, making fine-tuning efficient and affordable on a single Kaggle GPU.

### Base Model
- Qwen2.5-1.5B-Instruct

### Dataset
- CodeAlpaca-20k

### Fine-Tuning Method
- QLoRA
- 4-bit Quantization
- LoRA Rank = 16

### Goal
Improve the model's ability to answer coding-related questions while training less than 1% of total parameters.

In [1]:
!pip install -U transformers

## Remote Inference via Inference Providers 
Ensure you have a valid **HF_TOKEN** set in your environment. You can get your token from [your settings page](https://huggingface.co/settings/tokens). Note: running this may incur charges above the free tier.
The following Python example shows how to run the model remotely on HF Inference Providers, automatically selecting an available inference provider for you. 
For more information on how to use the Inference Providers, please refer to our [documentation and guides](https://huggingface.co/docs/inference-providers/en/index).

In [4]:
import os
os.environ['HF_TOKEN'] = 'hf_acess_key'

# Dataset Loading

The CodeAlpaca dataset contains instruction-response pairs focused on programming tasks such as:

- Python programming
- Algorithms
- Data structures
- SQL
- Debugging
- Software engineering concepts

This dataset serves as the domain-specific knowledge source for adaptation.

In [5]:
import pandas as pd

df = pd.read_json("hf://datasets/sahil2801/CodeAlpaca-20k/code_alpaca_20k.json")

In [6]:
print(df.head())
print(df.columns)

                                         instruction  \
0  Create an array of length 5 which contains all...   
1  Formulate an equation to calculate the height ...   
2  Write a replace method for a string class whic...   
3  Create an array of length 15 containing number...   
4  Write a function to find the number of distinc...   

                                               input  \
0                                                      
1                                                      
2  string = "Hello World!"\nreplace_with = "Greet...   
3                                                      
4  matrix = [[1, 0, 0],\n          [1, 0, 1],\n  ...   

                                              output  
0                             arr = [2, 4, 6, 8, 10]  
1  Height of triangle = opposite side length * si...  
2  def replace(self, replace_with):\n    new_stri...  
3  arr = [3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33...  
4  def find_num_distinct_states(matrix):\n    sta..

# Dataset Inspection

Before training, we inspect the dataset structure to understand:

- Available columns
- Instruction format
- Response format
- Dataset size

This helps verify that the data is suitable for instruction tuning.

In [7]:
def format_example(row):

    if str(row["input"]).strip():
        text = f"""### Instruction:
{row['instruction']}

### Input:
{row['input']}

### Response:
{row['output']}"""
    else:
        text = f"""### Instruction:
{row['instruction']}

### Response:
{row['output']}"""

    return text

df["text"] = df.apply(format_example, axis=1)

In [8]:
print(df["text"].iloc[0])

### Instruction:
Create an array of length 5 which contains all even numbers between 1 and 10.

### Response:
arr = [2, 4, 6, 8, 10]


In [9]:
df = df.sample(
    n=5000,
    random_state=42
).reset_index(drop=True)

# Train and Validation Split

The dataset is divided into:

- Training Set (90%)
- Validation Set (10%)

The training set is used for learning.

The validation set remains unseen during training and is used to evaluate generalization performance.

In [10]:
from datasets import Dataset

train_size = int(len(df) * 0.9)

train_df = df[:train_size]
eval_df = df[train_size:]

train_dataset = Dataset.from_pandas(train_df[["text"]])
eval_dataset = Dataset.from_pandas(eval_df[["text"]])

# Loading the Base Model

Qwen2.5-1.5B-Instruct is loaded as the foundation model.

This model already possesses strong general language and coding capabilities.

The objective is to specialize it further toward programming-related tasks.

In [2]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': "I am Qwen, a large language model created by Alibaba Cloud. I was trained on the massive amounts of text data available on the internet to understand human language and generate coherent responses to questions or prompts. My purpose is to assist with tasks such as answering questions, generating text based on input, and providing information on various topics. Please let me know if there's anything specific you'd like help with!"}]}]

# QLoRA Quantization

To reduce memory requirements, the model is loaded in 4-bit precision.

Benefits:

- Lower GPU memory usage
- Faster training
- Enables fine-tuning on commodity hardware

This approach is known as QLoRA.

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

messages = [
    {"role": "user", "content": "Who are you?"}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

I am Qwen, a large language model developed by Alibaba Cloud. I'm here to assist with various tasks and answer questions to the best of my ability based on the information available to me. If you have any queries or need help with something,


# Tokenization

Language models cannot directly process text.

The tokenizer converts text into numerical token IDs that can be understood by the model.

Labels are set equal to input IDs for causal language modeling.

In [11]:
MAX_LENGTH = 512

def tokenize_fn(example):
    out = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

    out["labels"] = out["input_ids"].copy()

    return out

In [12]:
train_tok = train_dataset.map(
    tokenize_fn,
    remove_columns=train_dataset.column_names
)

eval_tok = eval_dataset.map(
    tokenize_fn,
    remove_columns=eval_dataset.column_names
)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [13]:
print(train_tok[0].keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


# Low-Rank Adaptation (LoRA)

Instead of updating all model parameters, LoRA inserts small trainable adapter matrices into attention layers.

Configuration:

- Rank (r): 16
- Alpha: 32
- Dropout: 0.05

This dramatically reduces the number of trainable parameters while maintaining performance.

# Parameter Efficiency Analysis

After applying LoRA:

- Total Parameters: 1.55 Billion
- Trainable Parameters: 4.36 Million
- Trainable Percentage: 0.28%

This demonstrates the efficiency of parameter-efficient fine-tuning.

In [14]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


# Training Configuration

Training is performed using the Hugging Face Trainer API.

Key settings:

- Epochs: 2
- Learning Rate: 2e-4
- Batch Size: 2
- Gradient Accumulation: 4

These settings provide a balance between training speed and model performance.

In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./codealpaca_lora",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=True,
    report_to="none"
)

# Fine-Tuning Process

The model is trained on the CodeAlpaca dataset using QLoRA.

During training:

1. Predictions are generated.
2. Loss is computed against expected responses.
3. LoRA adapter weights are updated.
4. Validation loss is monitored after each epoch.

In [16]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok
)

In [17]:
print(len(train_dataset))
print(len(eval_dataset))

4500
500


In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.126955,0.125333
2,0.118596,0.124809


TrainOutput(global_step=1126, training_loss=0.13904237667790104, metrics={'train_runtime': 15654.1576, 'train_samples_per_second': 0.575, 'train_steps_per_second': 0.072, 'total_flos': 3.6348791095296e+16, 'train_loss': 0.13904237667790104, 'epoch': 2.0})

# Saving the LoRA Adapter

The trained adapter weights are saved for future inference.

This allows the adapted model to be reloaded without storing the full 1.55B parameter model.

In [21]:
trainer.save_model("./codealpaca_lora")
tokenizer.save_pretrained("./codealpaca_lora")

('./codealpaca_lora/tokenizer_config.json',
 './codealpaca_lora/chat_template.jinja',
 './codealpaca_lora/tokenizer.json')

# Training Results

Final Metrics

| Metric | Value |
|----------|----------|
| Training Loss | 0.1186 |
| Validation Loss | 0.1248 |
| Trainable Parameters | 4.36M |
| Total Parameters | 1.55B |
| Trainable % | 0.28% |

The close alignment between training and validation loss suggests good generalization without significant overfitting.

In [22]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Epoch
0.118596,0.124809,2


{'eval_loss': 0.12480874359607697}


# Conclusion

This project successfully adapted Qwen2.5-1.5B-Instruct to the coding domain using QLoRA.

Key achievements:

- Fine-tuned using only 0.28% of total parameters
- Trained on CodeAlpaca programming data
- Achieved low validation loss
- Demonstrated parameter-efficient domain adaptation

The results show that QLoRA is an effective approach for specializing large language models on domain-specific tasks while keeping computational requirements low.

In [23]:
from transformers import AutoTokenizer, AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

base_tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [24]:
from peft import PeftModel

ft_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

ft_model = PeftModel.from_pretrained(
    ft_model,
    "./codealpaca_lora"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [26]:
def generate(model, tokenizer, prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [31]:
prompt = "Explain recursion with an example."

print("BASE")
print(generate(base_model, base_tokenizer, prompt))

print("FINE-TUNED")
print(generate(ft_model, base_tokenizer, prompt))

BASE
Explain recursion with an example. Recursion is a programming technique where a function calls itself repeatedly until it reaches a base case that stops the recursive process. The purpose of recursion is to solve complex problems by breaking them down into smaller, more manageable sub-problems.

Here's an example in Python:

```python
def factorial(n):
    if n == 1:
        return 1
    else:
        return n * factorial(n-1)

print(factorial(5))  # Output: 120
```

In this example, the `factorial` function calculates the factorial of a given number `n`. The factorial of a number is defined as the product of all positive integers less than or equal to that number. For instance, the factorial of 5 (written as 5!) is calculated as follows:

5! = 5 × 4 × 3 × 2 × 1 = 120

The `factorial` function uses recursion to calculate the factorial. It checks
FINE-TUNED
Explain recursion with an example. Recursion is a programming technique in which a function calls itself repeatedly until it r

In [32]:
prompts = [
    "Write a Python function to merge two sorted arrays.",
    "Write a SQL query to find the second highest salary.",
    "Implement binary search in Python.",
    "Write a Python function to detect a cycle in a linked list.",
    "Write code for breadth first search."
]

for prompt in prompts:
    print("=" * 100)
    print(f"PROMPT: {prompt}\n")

    print("BASE MODEL:")
    print(generate(base_model, base_tokenizer, prompt))

    print("\nFINE-TUNED MODEL:")
    print(generate(ft_model, base_tokenizer, prompt))

    print("\n")

PROMPT: Write a Python function to merge two sorted arrays.

BASE MODEL:
Write a Python function to merge two sorted arrays. The function should take three parameters: the first array, the second array, and an integer `n` representing the number of elements in the first array that are not yet included in the merged result. The function should return a single sorted list containing all elements from both input arrays up to index `n`.

For example:
```python
arr1 = [2, 3, 5]
arr2 = [1, 4]
merge_arrays(arr1, arr2, 2) # returns [1, 2, 3, 4, 5]
```

In this case, we start with one element from `arr1` (which is `[2]`) and another element from `arr2` (which is `[1]`). We then add the next available element from each array until we have processed `n=2` elements from `arr1`. ```python
def merge_sorted_arrays(arr1, arr2, n):
    """
    Merge two sorted

FINE-TUNED MODEL:
Write a Python function to merge two sorted arrays. def merge_arrays(arr1, arr2):
    # Initialize the result array
    merge